In [1]:
import os
import json
import shutil
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [6]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/ruwiki_good.txt',
)

dataset.get_possible_modalities()

{'@categories', '@lemmatized', '@ngramms'}

In [7]:
MAIN_MODALITY = '@lemmatized'

In [8]:
dataset._data.head()

,vw_text,raw_text,id
id,,,
Санкт-Петербург,Санкт-Петербург |@lemmatized год:301 петроград...,,Санкт-Петербург
Дворцовая_площадь,Дворцовая_площадь |@lemmatized дворцовый:43 пл...,,Дворцовая_площадь
Греко-персидские_войны,Греко-персидские_войны |@lemmatized грёкий:23 ...,,Греко-персидские_войны
Тихий_океан,Тихий_океан |@lemmatized тихий:92 океан:174 ус...,,Тихий_океан
Атлантический_океан,Атлантический_океан |@lemmatized атлантический...,,Атлантический_океан


In [9]:
def is_good(coherence):
    return 0.9918260892662304 <= coherence

def is_bad(coherence):
    return coherence <= 0.5677332839669632

In [15]:
dataset.get_dictionary()

artm.Dictionary(name=12a8a678-44e6-4fe2-b752-7a647b1dff1e, num_entries=892938)

In [16]:
dictionary = dataset.get_dictionary()

In [17]:
print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=12a8a678-44e6-4fe2-b752-7a647b1dff1e, num_entries=892938)


In [18]:
dictionary.filter(min_df=5, max_df_rate=0.5)

artm.Dictionary(name=12a8a678-44e6-4fe2-b752-7a647b1dff1e, num_entries=61688)

In [19]:
dataset._cached_dict = dictionary

In [20]:
dataset.get_dictionary()

artm.Dictionary(name=12a8a678-44e6-4fe2-b752-7a647b1dff1e, num_entries=61688)

In [21]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [22]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 47.6 s, sys: 1.26 s, total: 48.9 s
Wall time: 48.4 s


In [23]:
co_occurences.shape

(61688, 61688)

In [24]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [25]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [26]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [29]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 12  # Changed here 10 ** 9

    def __init__(self, name: str, topic_names: List[str], parent_model=None, parent_phi=None):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._parent_model = parent_model
        self._parent_phi = parent_phi

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)

        if self._parent_phi is not None:
            parent_phi = self._parent_phi
            vals = parent_phi.values
        else:
            parent_phi = self._parent_model.get_phi()
            vals = parent_phi.values[:, self._topic_indices]

        assert vals.shape[0] == rwt.shape[0]
        assert vals.shape[1] == len(self._topic_indices)
        
        rwt[:, self._topic_indices] += vals

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [30]:
class DecorrelatorWithOtherPhiRegularizer(BaseRegularizer):
    def __init__(self, name, tau, topic_names, other_phi):
        super().__init__(name, tau=tau)

        self._topic_names = topic_names
        self._other_phi = other_phi
        self._other_topic_sum = self._other_phi.values.sum(
            axis=1, keepdims=True
        )
        
        self._topic_indices = None
        
    def grad(self, pwt, nwt):
        # print('Decorring')

        rwt = np.zeros_like(pwt)
        rwt[:, self._topic_indices] += (
            pwt.values[:, self._topic_indices] * self._other_topic_sum
        )

        return -1 * self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [31]:
class DecorrelatorWithOtherPhiRegularizer2(BaseRegularizer):
    def __init__(self, name, tau, topic_names, other_phi, num_iters: Optional[int] = None):
        super().__init__(name, tau=tau)

        self._topic_names = topic_names
        self._other_phi = other_phi
        self._num_iters = num_iters
        self._cur_iter = 0
        
        self._topic_indices = None
        
    def grad(self, pwt, nwt):
        rwt = np.zeros_like(pwt)
        
        if self._num_iters is not None and self._cur_iter >= self._num_iters:
            return rwt

        correlations = cdist(
            self._other_phi.values.T,
            pwt.values[:, self._topic_indices].T,
            lambda u, v: (u * v).sum()
        )
        weighted_other_topics = self._other_phi.values.dot(correlations)

        rwt[:, self._topic_indices] += (
            pwt.values[:, self._topic_indices] * weighted_other_topics
        )
        self._cur_iter += 1

        return -1 * self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [32]:
NUM_TOPICS = 50  # vary
MAX_NUM_TRAINS = 20
NUM_ITERATIONS = 10  # 20
NUM_TOP_TOKENS = 20

In [33]:
NUM_GOOD_TOPICS_THRESHOLD = NUM_TOPICS - 5

In [34]:
NUM_GOOD_TOPICS_THRESHOLD

45

In [35]:
def fit_and_compute_scores(model, dataset, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [36]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [37]:
NUM_TRAINS = 3
TOPIC_INDICES = list(range(NUM_TOPICS))

In [38]:
TOPIC_INDICES

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49]

In [39]:
! ls results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [40]:
! ls results/ruwikigood/

ablation_study			     iterative2_1000000000
decorrelation.json		     iterative2_10000000000
iterative_1000000		     iterative2_100000000000
iterative_10000000		     iterative2_100000000000.json
iterative_100000000		     iterative2_10000000000.json
iterative_100000000_unfinished.json  iterative2_1000000000.json
_iterative_10000000.json	     lda.json
iterative_10000000.json		     plsa.json
_iterative_1000000.json		     sparse.json
iterative_1000000.json		     tless.json


In [69]:
! tail -n 50 results/ruwikigood/iterative2_10000000000.json

            "17": 1.009062903434669,
            "18": 1.4248513588351155,
            "19": 0.987757023282529
        },
        "num_topics": {
            "good": 16,
            "bad": 0,
            "not_good": 4,
            "total_bad": 12
        }
    },
    {
        "scores": {
            "perplexity": 5457.533203125,
            "coherence_20": 1.0475442204922938,
            "diversity_euclidean": 0.07185180809384983,
            "diversity_jensenshannon": 0.730655811558549,
            "diversity_hellinger": 0.8571272283587976,
            "diversity_cosine": 0.919584714192535
        },
        "topic_coherences": {
            "0": 1.1117905194294229,
            "1": 0.9505906411745993,
            "2": 0.7534125513845712,
            "3": 1.0768358454454414,
            "4": 0.9156432997193535,
            "5": 0.8853331691061566,
            "6": 1.0297691355246918,
            "7": 0.8975712771845624,
            "8": 0.9395098499846036,
            "9": 1.17566160

In [71]:
# BEST_TAUS = [100000,    100000000]
BEST_TAUS =   [1000000,   10000000000]

In [72]:
! ls results50

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [73]:
SAVE_FOLDER = 'results50/ruwikigood'

os.makedirs(SAVE_FOLDER, exist_ok=True)

In [74]:
SAVE_FOLDER

'results50/ruwikigood'

In [75]:
BEST_TAUS

[1000000, 10000000000]

In [76]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

DECORRELATION_TAUS = [BEST_TAUS[0]]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    
    res_file_path = SAVE_FOLDER + f'/iterative_{int(key)}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue
    
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        
        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:

            print('test_1')

            if good_phi is None:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
            else:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_phi=good_phi,
                    topic_names=good_topic_names,
                )

            print('test_2')
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            print('test_3')
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )

            print('test_4')
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )

            print('test_5')
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            print('test_6')
            
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }

            print('test_7')
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break

            print('test_8')
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            del prev_model
            
            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

1000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.1885551253311596
sparse_theta_sp: -1.3520386576111325
decorrelation: 0.01
None
num_topics: {'good': 12, 'bad': 10, 'not_good': 38, 'total_bad': 10}
1
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8f4995f490>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f9018e73b20>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f9018e739a0>}
test_8
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.2480988491199468
sparse_theta_sp: -1.7789982336988581
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 1000000
ext_decorr_good: 1000000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 21, 'bad': 3, 'not_good': 29, 'total_bad': 13}
Removing: results50/ruwikigood/iterative_1000000/0
2
test_1
test_2
test_3
test_4
test_5


...

test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8fee925e20>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f90086a3eb0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f8fee925a60>}
test_8
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.8570687515052707
sparse_theta_sp: -6.145630261868784
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 1000000
ext_decorr_good: 1000000
SUCCESS: more good topics
SUCCESS: less bad topics
SUCCESS: no bad topics!
num_topics: {'good': 43, 'bad': 0, 'not_good': 7, 'total_bad': 26}
Removing: results50/ruwikigood/iterative_1000000/4
6
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8ffd3dd250>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f90086a3f10>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f8ffd3ddee0>}
test_8
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -1.346822323793997
sparse_theta_sp: -9.657418982936658
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 1000000
ext_decorr_good: 1000000
DOWNFALL: more bad topics...
num_topics: {'good': 43, 'bad': 7, 'not_good': 7, 'total_bad': 33}
Removing: results50/ruwikigood/iterative_1000000/5
7
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f900881d850>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f8ffd0581c0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f8ffd3ddf70>}
Skipping computation of dists: (1081, 1225).
Skipping computation of dists: (1081, 1225).
Early stopping because of corrupted topics
Saving results


In [77]:
1

1

In [78]:
results.keys()

dict_keys([1000000])

In [79]:
SAVE_FOLDER

'results50/ruwikigood'

In [80]:
! ls $SAVE_FOLDER

decorrelation.json  iterative_1000000.json  plsa.json	 tless.json
iterative_1000000   lda.json		    sparse.json


In [81]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

DECORRELATION_TAUS = [BEST_TAUS[1]]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    
    res_file_path = SAVE_FOLDER + f'/iterative2_{int(key)}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue
    
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        
        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:

            print('test_1')

            if good_phi is None:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
            else:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_phi=good_phi,
                    topic_names=good_topic_names,
                )

            print('test_2')
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
            
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            del prev_model
            
            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative2_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

10000000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.1885551253311596
sparse_theta_sp: -1.3520386576111325
decorrelation: 0.01
None
num_topics: {'good': 12, 'bad': 10, 'not_good': 38, 'total_bad': 10}
1
test_1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9008585850>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9008585b20>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9008585d30>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.2480988491199468
sparse_theta_sp: -1.7789982336988581
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 10000000000
ext_decorr_good: 10000000000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 26, 'bad': 2, 'not_good': 24, 'total_bad': 12}
Removing: results50/ruwikigood/iterative2_10000000000/0
2
test_1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8ffd35f160>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9018b80dc0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9008585760>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.3928231777732491
sparse_theta_sp: -2.8167472033565257
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 10000000000
ext_decorr_good: 10000000000
SUCCESS: more good topics
SUCCESS: less bad topics
SUCCESS: no bad topics!
num_topics: {'good': 40, 'bad': 0, 'not_good': 10, 'total_bad': 12}
Removing: results50/ruwikigood/iterative2_10000000000/1
3
test_1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8ff3ec4190>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9008585790>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f8ff3ec4460>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.9427756266557978
sparse_theta_sp: -6.760193288055662
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 10000000000
ext_decorr_good: 10000000000
SUCCESS: more good topics
SUCCESS: no bad topics!
num_topics: {'good': 48, 'bad': 0, 'not_good': 2, 'total_bad': 12}
Removing: results50/ruwikigood/iterative2_10000000000/2
Saving results


In [82]:
results.keys()

dict_keys([10000000000])

In [83]:
! ls $SAVE_FOLDER

decorrelation.json	iterative2_10000000000	     plsa.json
iterative_1000000	iterative2_10000000000.json  sparse.json
iterative_1000000.json	lda.json		     tless.json


## Ablation Study

In [84]:
DECORRELATION_TAU = BEST_TAUS[0]

ALL_PARAMS = [
    # (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    # (0, 1, 0),
    # (0, 0, 1),
]

In [85]:
SAVE_FOLDER + f'/ablation_study'

'results50/ruwikigood/ablation_study'

In [86]:
os.makedirs(SAVE_FOLDER + f'/ablation_study', exist_ok=True)

In [87]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    good_phi = None
    seed = 0

    os.makedirs(
        os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}'),
        exist_ok=True
    )

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            print(f'Loading seed results from "{seed_save_folder}".')
            
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:
            # assert False
            
            custom_regularizers = dict()
            
            if params[0]:
                if good_phi is None:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_model=prev_model._model,
                        topic_names=good_topic_names,
                    )
                else:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_phi=good_phi,
                        topic_names=good_topic_names,
                    )
    
                print('test_2')
                
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            if params[1]:
                bad_phi = deepcopy(bad_phi)
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None

            if good_phi is None:
                good_phi = prev_model._model.get_phi()[good_topic_names]
    
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.1885551253311596
sparse_theta_sp: -1.3520386576111325
decorrelation: 0.01
None
num_topics: {'good': 12, 'bad': 10, 'not_good': 38, 'total_bad': 10}
1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8f0f67c5e0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f8ff3ec4190>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.2480988491199468
sparse_theta_sp: -1.7789982336988581
decorrelation: 0.01
fix: 1000000000000
ext_decorr_good: 1000000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 19, 'bad': 5, 'not_good': 31, 'total_bad': 15}
Removing: results50/ruwikigood/ablation_study/iterative_1000000_1-0-1/0
2
test_2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8ffdf5a910>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.5892347666598737
sparse_theta_sp: -4.225120805034789
decorrelation: 0.01
fix: 1000000000000
DOWNFALL: more bad topics...
num_topics: {'good': 34, 'bad': 10, 'not_good': 16, 'total_bad': 144}
Removing: results50/ruwikigood/ablation_study/iterative_1000000_1-0-0/16
18
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9008192460>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.5892347666598737
sparse_theta_sp: -4.225120805034789
decorrelation: 0.01
fix: 1000000000000
SUCCESS: less bad topics
num_topics: {'good': 34, 'bad': 7, 'not_good': 16, 'total_bad': 151}
Removing: results50/ruwikigood/ablation_study/iterative_1000000_1-0-0/17
19
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8ff3f1f700>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.5892347666598737
sparse_theta_sp: -4.225120805034789
decorrelation: 0.01
fix: 1000000000000
DOWNFALL: more bad topics...
num_topics: {'good': 34, 'bad': 8, 'not_good': 16, 'total_bad': 159}
Removing: results50/ruwikigood/ablation_study/iterative_1000000_1-0-0/18
Saving results


In [88]:
1

1

In [89]:
DECORRELATION_TAU = BEST_TAUS[1]

In [90]:
DECORRELATION_TAU

10000000000

In [91]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    good_phi = None
    seed = 0

    os.makedirs(
        os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}'),
        exist_ok=True
    )

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            print(f'Loading seed results from "{seed_save_folder}".')
            
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:
            # assert False
            
            custom_regularizers = dict()
            
            if params[0]:
                if good_phi is None:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_model=prev_model._model,
                        topic_names=good_topic_names,
                    )
                else:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_phi=good_phi,
                        topic_names=good_topic_names,
                    )
    
                print('test_2')
                
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            if params[1]:
                bad_phi = deepcopy(bad_phi)
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None

            if good_phi is None:
                good_phi = prev_model._model.get_phi()[good_topic_names]
    
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.1885551253311596
sparse_theta_sp: -1.3520386576111325
decorrelation: 0.01
None
num_topics: {'good': 12, 'bad': 10, 'not_good': 38, 'total_bad': 10}
1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8f0f673f70>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f8ff3f1f700>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.2480988491199468
sparse_theta_sp: -1.7789982336988581
decorrelation: 0.01
fix: 1000000000000
ext_decorr_good: 10000000000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 21, 'bad': 4, 'not_good': 29, 'total_bad': 14}
Removing: results50/ruwikigood/ablation_study/iterative2_10000000000_1-0-1/0
2
test_2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8ffdca3190>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.5892347666598737
sparse_theta_sp: -4.225120805034789
decorrelation: 0.01
fix: 1000000000000
DOWNFALL: more bad topics...
num_topics: {'good': 34, 'bad': 10, 'not_good': 16, 'total_bad': 144}
Removing: results50/ruwikigood/ablation_study/iterative2_10000000000_1-0-0/16
18
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8ffd5000a0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.5892347666598737
sparse_theta_sp: -4.225120805034789
decorrelation: 0.01
fix: 1000000000000
SUCCESS: less bad topics
num_topics: {'good': 34, 'bad': 7, 'not_good': 16, 'total_bad': 151}
Removing: results50/ruwikigood/ablation_study/iterative2_10000000000_1-0-0/17
19
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f8f4a32fee0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.5892347666598737
sparse_theta_sp: -4.225120805034789
decorrelation: 0.01
fix: 1000000000000
DOWNFALL: more bad topics...
num_topics: {'good': 34, 'bad': 8, 'not_good': 16, 'total_bad': 159}
Removing: results50/ruwikigood/ablation_study/iterative2_10000000000_1-0-0/18
Saving results


In [94]:
1

1

In [95]:
results.keys()

dict_keys([(1, 0, 1), (1, 1, 0), (1, 0, 0)])